# Práctica 4 — Expresión diferencial con PyDESeq2: TNF en células T3M4

**Máster en Bioinformática — Módulo de Transcriptómica**

## Objetivo

Analizar los cambios transcriptómicos inducidos por **TNF** en células **T3M4** de
cáncer de páncreas, identificando genes diferencialmente expresados y explorando su
posible interpretación biológica.

## Diseño experimental

| Muestra | Condición | Réplica |
|---|---|---:|
| shCtrl_Untreated_rep1 | Untreated | 1 |
| shCtrl_Untreated_rep2 | Untreated | 2 |
| shCtrl_Untreated_rep3 | Untreated | 3 |
| shCtrl_TNF_rep1 | TNF | 1 |
| shCtrl_TNF_rep2 | TNF | 2 |
| shCtrl_TNF_rep3 | TNF | 3 |

Los datos proceden de **featureCounts** y se cargarán primero en un objeto **AnnData**.
Después aplicaremos **PyDESeq2** para normalización, análisis exploratorio y test de
expresión diferencial.

> **Nota:** la matriz completa del proyecto contiene más muestras (shBPTF y una cuarta
> réplica). Para este ejercicio usamos únicamente las **6 muestras shCtrl** indicadas
> en el diseño experimental.

## 0. Instalación

Instalamos `pydeseq2` y fijamos `pandas==2.2.3` por compatibilidad con el entorno de
Google Colab. La instalación puede tardar 1–2 minutos.

In [ ]:
# ============================================================================
# CELDA 0: Instalación e importaciones
# ============================================================================
!pip install -q "pydeseq2==0.5.4" "pandas==2.2.3"

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference
from sklearn.decomposition import PCA

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 20)

print("✓ Entorno listo")

## 1. Obtención de la matriz de conteos

Por defecto el cuaderno descarga `counts_final_example.csv` desde el repositorio del
curso. Si todavía no está subido a GitHub, cambia `MODO = "SUBIR"` y selecciona el
archivo desde tu ordenador.

También puedes usar la matriz completa `counts_final.tsv`: el cuaderno seleccionará
automáticamente las 6 muestras del diseño.

In [ ]:
# ============================================================================
# CELDA 1: Descarga o subida de datos
# ============================================================================
import urllib.request

MODO = "URL"   # "URL" | "SUBIR"

if MODO == "URL":
    DATA_DIR = Path("/content/datos_tnf_t3m4")
    DATA_DIR.mkdir(exist_ok=True)
    counts_path = DATA_DIR / "counts_final_example.csv"
    URL_COUNTS = "https://raw.githubusercontent.com/victorsanchezarevalo/transcriptomica-master/main/practicas/04-tnf-t3m4-pydeseq2/datos/counts_final_example.csv"
    if not counts_path.exists():
        urllib.request.urlretrieve(URL_COUNTS, counts_path)
    print(f"✓ Datos descargados: {counts_path}")

elif MODO == "SUBIR":
    from google.colab import files
    DATA_DIR = Path("/content")
    print("Sube counts_final_example.csv (o counts_final.tsv):")
    files.upload()
    if (DATA_DIR / "counts_final_example.csv").exists():
        counts_path = DATA_DIR / "counts_final_example.csv"
    elif (DATA_DIR / "counts_final.tsv").exists():
        counts_path = DATA_DIR / "counts_final.tsv"
    else:
        raise FileNotFoundError("No se encontró counts_final_example.csv ni counts_final.tsv")

OUT_DIR = Path("/content/tnf_t3m4_results")
OUT_DIR.mkdir(exist_ok=True)
print(f"✓ Resultados se guardarán en: {OUT_DIR}")

---
# PARTE I — Carga de datos y creación del objeto AnnData

`AnnData` organiza la matriz de conteos junto con los metadatos de las muestras
(`obs`) y de los genes (`var`). En este formato, las **muestras son filas** y los
**genes son columnas**.

In [ ]:
# ============================================================================
# CELDA 2: Carga, validación y creación de AnnData
# ============================================================================

# Diseño experimental del ejercicio
MUESTRAS = [
    "shCtrl_Untreated_rep1", "shCtrl_Untreated_rep2", "shCtrl_Untreated_rep3",
    "shCtrl_TNF_rep1", "shCtrl_TNF_rep2", "shCtrl_TNF_rep3",
]

# Leer CSV o TSV según la extensión
sep = "\t" if counts_path.suffix == ".tsv" else ","
counts_raw = pd.read_csv(counts_path, sep=sep, index_col=0)

# Quitar posibles versiones de Ensembl (ENSG... .12 → ENSG...)
counts_raw.index = counts_raw.index.astype(str).str.split(".").str[0]
counts_raw = counts_raw.groupby(level=0).sum()

# Seleccionar solo las 6 muestras del diseño, en el orden previsto
faltan = [s for s in MUESTRAS if s not in counts_raw.columns]
if faltan:
    raise ValueError(f"Faltan estas muestras en la matriz: {faltan}")
counts = counts_raw.loc[:, MUESTRAS].copy()

# Validaciones básicas de una matriz featureCounts
if counts.isna().sum().sum() > 0:
    raise ValueError("La matriz contiene valores NA")
if (counts < 0).sum().sum() > 0:
    raise ValueError("La matriz contiene conteos negativos")
if not np.allclose(counts.to_numpy(), np.round(counts.to_numpy())):
    raise ValueError("Los conteos deberían ser números enteros")
counts = counts.astype(int)

# Metadatos de muestras
metadata = pd.DataFrame(index=MUESTRAS)
metadata["sample"] = MUESTRAS
metadata["treatment"] = ["Untreated" if "_Untreated_" in s else "TNF" for s in MUESTRAS]
metadata["replicate"] = [s.rsplit("_rep", 1)[1] for s in MUESTRAS]
metadata["treatment"] = pd.Categorical(
    metadata["treatment"], categories=["Untreated", "TNF"], ordered=True
)

# Metadatos de genes
var = pd.DataFrame(index=counts.index)
var["gene_id"] = counts.index

# AnnData: muestras × genes
adata = ad.AnnData(
    X=counts.T.to_numpy(dtype=np.int32),
    obs=metadata,
    var=var,
)

print("Matriz original:", counts.shape[0], "genes ×", counts.shape[1], "muestras")
print("\nMetadatos:")
display(adata.obs)
print("\nProfundidad de secuenciación (conteos totales):")
display(counts.sum(axis=0).to_frame("total_counts"))

# Filtrado mínimo: eliminar genes prácticamente no detectados
genes_to_keep = np.asarray(adata.X.sum(axis=0)).ravel() >= 10
adata = adata[:, genes_to_keep].copy()

print(f"Genes tras filtrar (suma de conteos ≥ 10): {adata.n_vars}")
print(f"AnnData final: {adata.n_obs} muestras × {adata.n_vars} genes")

---
# PARTE II — Actividad 6.1: Normalización con DESeq2

PyDESeq2 implementa el modelo estadístico de DESeq2. La normalización por factores de
tamaño corrige diferencias de profundidad de secuenciación entre muestras.

In [ ]:
# ============================================================================
# CELDA 3: Modelo DESeq2, normalización y ajuste
# ============================================================================

inference = DefaultInference(n_cpus=2)

dds = DeseqDataSet(
    adata=adata,
    design="~treatment",       # Untreated es la referencia por el orden categórico
    refit_cooks=True,
    inference=inference,
)

# Ajusta factores de tamaño, dispersiones y modelo GLM
dds.deseq2()
print(dds)

# Factores de tamaño estimados
size_factors = dds.obs["size_factors"]
print("\nFactores de tamaño:")
display(size_factors.to_frame("size_factor"))

# Conteos normalizados por tamaño de librería
normalized_counts = pd.DataFrame(
    dds.X / size_factors.to_numpy()[:, None],
    index=dds.obs_names,
    columns=dds.var_names,
).T

normalized_counts.to_csv(OUT_DIR / "conteos_normalizados_deseq2.csv")
print("✓ Conteos normalizados guardados en conteos_normalizados_deseq2.csv")
display(normalized_counts.head())

> ❓ **Preguntas 6.1:**
> 1. ¿Por qué es necesaria la normalización en RNA-seq?
> 2. ¿Qué método de normalización usa DESeq2 por defecto?
> 3. ¿Los factores de tamaño son muy distintos entre muestras? ¿Qué indicaría un factor
>    muy alejado de 1?

---
# PARTE III — Actividad 6.2: Análisis exploratorio

Aplicamos la **VST** (*Variance Stabilizing Transformation*) para visualización. La VST
no se usa para el test estadístico; sirve para que PCA y correlaciones no estén
dominadas por la dependencia media–varianza de los conteos.

In [ ]:
# ============================================================================
# CELDA 4: Transformación VST
# ============================================================================

dds.vst(use_design=False, fit_type="mean")
vst_counts = dds.layers["vst_counts"]

vst_df = pd.DataFrame(
    vst_counts,
    index=dds.obs_names,
    columns=dds.var_names,
)

print("Matriz VST:", vst_df.shape[0], "muestras ×", vst_df.shape[1], "genes")
vst_df.to_csv(OUT_DIR / "vst_counts.csv")
display(vst_df.iloc[:, :5].head())

In [ ]:
# ============================================================================
# CELDA 5: PCA de las muestras
# ============================================================================

pca = PCA(n_components=2)
pca_results = pca.fit_transform(vst_df)

pca_df = pd.DataFrame(pca_results, columns=["PC1", "PC2"], index=vst_df.index)
pca_df["treatment"] = dds.obs["treatment"].astype(str)
pca_df["replicate"] = dds.obs["replicate"].astype(str)
pca_df["sample"] = pca_df.index

fig, ax = plt.subplots(figsize=(9, 7))
sns.scatterplot(
    data=pca_df,
    x="PC1", y="PC2",
    hue="treatment",
    style="replicate",
    s=140,
    palette={"Untreated": "#4C72B0", "TNF": "#DD8452"},
    edgecolor="black",
    linewidth=0.5,
    ax=ax,
)

for _, row in pca_df.iterrows():
    ax.annotate(row["sample"], (row["PC1"], row["PC2"]),
                xytext=(6, 6), textcoords="offset points", fontsize=8)

ax.set_title("PCA de muestras T3M4 (datos VST)", fontweight="bold")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)")
plt.tight_layout()
plt.savefig(OUT_DIR / "PCA_muestras.png", dpi=150)
plt.show()

print(f"Varianza explicada PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"Varianza explicada PC2: {pca.explained_variance_ratio_[1]:.1%}")

In [ ]:
# ============================================================================
# CELDA 6: Heatmap de correlación entre muestras
# ============================================================================

corr = vst_df.T.corr(method="pearson")

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".3f",
    cmap="vlag",
    vmin=max(0.90, corr.min().min()),
    vmax=1.0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlación de Pearson"},
    ax=ax,
)
ax.set_title("Correlación entre muestras (datos VST)", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_correlacion_muestras.png", dpi=150)
plt.show()

> ❓ **Preguntas 6.2:**
> 1. ¿Se agrupan las muestras según su condición experimental?
> 2. ¿Hay alguna muestra que parezca outlier?
> 3. ¿Por qué usamos VST antes del PCA en lugar de conteos crudos?

---
# PARTE IV — Actividad 6.3: Identificación de genes diferencialmente expresados

El contraste se define como **TNF / Untreated**:

- `log2FoldChange > 0`: más expresión tras TNF.
- `log2FoldChange < 0`: menos expresión tras TNF.

In [ ]:
# ============================================================================
# CELDA 7: Test de expresión diferencial TNF vs Untreated
# ============================================================================

ds = DeseqStats(
    dds,
    contrast=["treatment", "TNF", "Untreated"],
    inference=inference,
)

ds.summary()  # Test de Wald + corrección Benjamini-Hochberg
results = ds.results_df.copy()
results.index.name = "gene_id"

results.to_csv(OUT_DIR / "resultados_TNF_vs_Untreated.csv")
print("✓ Resultados guardados en resultados_TNF_vs_Untreated.csv")
display(results.sort_values("padj").head(15))

In [ ]:
# ============================================================================
# CELDA 8: Filtrado por FDR y magnitud del cambio
# ============================================================================

FDR = 0.05
LFC = 1.0

sig_mask = (results["padj"] < FDR) & (results["log2FoldChange"].abs() > LFC)
up_mask = sig_mask & (results["log2FoldChange"] > 0)
down_mask = sig_mask & (results["log2FoldChange"] < 0)

resumen_de = pd.Series({
    "Genes testeados": int(results["padj"].notna().sum()),
    "Genes con padj NA": int(results["padj"].isna().sum()),
    f"DE (FDR < {FDR}, |log2FC| > {LFC})": int(sig_mask.sum()),
    "Sobre-expresados en TNF": int(up_mask.sum()),
    "Sub-expresados en TNF": int(down_mask.sum()),
})
display(resumen_de.to_frame("n"))

resultados_sig = results.loc[sig_mask].sort_values("padj")
resultados_sig.to_csv(OUT_DIR / "genes_DE_FDR005_LFC1.csv")
print("✓ Genes DE guardados en genes_DE_FDR005_LFC1.csv")
display(resultados_sig.head(20))

> ❓ **Preguntas 6.3:**
> 1. ¿Cuántos genes están significativamente diferencialmente expresados?
> 2. ¿Cuántos están sobre-expresados y cuántos sub-expresados?
> 3. ¿Por qué algunos genes tienen `padj` igual a NA?

---
# PARTE V — Actividad 6.4: Visualización de resultados

In [ ]:
# ============================================================================
# CELDA 9: MA plot con la función de PyDESeq2
# ============================================================================

ds.plot_MA(s=8, alpha=0.4)
plt.savefig(OUT_DIR / "MA_plot_pydeseq2.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================================
# CELDA 10: Volcano plot con seaborn
# ============================================================================

plot_df = results.dropna(subset=["padj"]).copy()

# Evitar -log10(0) cuando padj es extremadamente pequeño y se redondea a 0
padj_positivos = plot_df.loc[plot_df["padj"] > 0, "padj"]
floor = padj_positivos.min() / 10 if len(padj_positivos) else 1e-300
plot_df["neglog10_padj"] = -np.log10(plot_df["padj"].clip(lower=floor))

plot_df["Categoría"] = "No significativo"
plot_df.loc[(plot_df["padj"] < FDR) & (plot_df["log2FoldChange"] > LFC), "Categoría"] = "Up en TNF"
plot_df.loc[(plot_df["padj"] < FDR) & (plot_df["log2FoldChange"] < -LFC), "Categoría"] = "Down en TNF"

fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(
    data=plot_df,
    x="log2FoldChange",
    y="neglog10_padj",
    hue="Categoría",
    palette={
        "No significativo": "lightgrey",
        "Up en TNF": "#DD8452",
        "Down en TNF": "#4C72B0",
    },
    s=12,
    alpha=0.65,
    linewidth=0,
    ax=ax,
)

ax.axvline(-LFC, color="black", linestyle="--", linewidth=0.8)
ax.axvline(LFC, color="black", linestyle="--", linewidth=0.8)
ax.axhline(-np.log10(FDR), color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("log2 Fold Change (TNF / Untreated)")
ax.set_ylabel("-log10(FDR)")
ax.set_title("Volcano plot — TNF vs Untreated", fontweight="bold")

# Etiquetar los 10 genes con menor FDR
for gene, row in plot_df.sort_values("padj").head(10).iterrows():
    ax.annotate(gene, (row["log2FoldChange"], row["neglog10_padj"]),
                xytext=(4, 4), textcoords="offset points", fontsize=7)

plt.tight_layout()
plt.savefig(OUT_DIR / "volcano_TNF_vs_Untreated.png", dpi=150)
plt.show()

In [ ]:
# ============================================================================
# CELDA 11: Heatmap de los 50 genes con menor FDR
# ============================================================================

top50 = results.dropna(subset=["padj"]).sort_values("padj").head(50).index
vst_top50 = vst_df.loc[:, top50].T  # genes × muestras

# Z-score por gen para visualizar patrones relativos entre muestras
zscores = vst_top50.sub(vst_top50.mean(axis=1), axis=0)
zscores = zscores.div(vst_top50.std(axis=1).replace(0, np.nan), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(8, 12))
g = sns.heatmap(
    zscores,
    cmap="vlag",
    center=0,
    yticklabels=True,
    xticklabels=True,
    cbar_kws={"label": "Z-score por gen"},
    ax=ax,
)
g.set_title("Top 50 genes diferencialmente expresados", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_top50_DE.png", dpi=150)
plt.show()

In [ ]:
# ============================================================================
# CELDA 12: Comprobación biológica — genes canónicos de respuesta a TNF
# ============================================================================

GENES_TNF = {
    "TNFAIP3": "ENSG00000118503",
    "NFKBIA": "ENSG00000100906",
    "IL1B": "ENSG00000125538",
    "BIRC3": "ENSG00000023445",
    "CXCL8": "ENSG00000169429",
    "CCL2": "ENSG00000108691",
    "ICAM1": "ENSG00000090339",
    "TNFAIP6": "ENSG00000123610",
    "NFKB1": "ENSG00000109320",
    "RELB": "ENSG00000104856",
}

filas = []
for symbol, gene_id in GENES_TNF.items():
    if gene_id in results.index:
        row = results.loc[gene_id]
        filas.append({
            "gene": symbol,
            "gene_id": gene_id,
            "baseMean": row["baseMean"],
            "log2FC": row["log2FoldChange"],
            "padj": row["padj"],
        })
    else:
        filas.append({"gene": symbol, "gene_id": gene_id,
                      "baseMean": np.nan, "log2FC": np.nan, "padj": np.nan})

respuesta_tnf = pd.DataFrame(filas)
display(respuesta_tnf)
respuesta_tnf.to_csv(OUT_DIR / "genes_canonicos_TNF.csv", index=False)

> ❓ **Preguntas 6.4:**
> 1. ¿Qué patrones observas en el MA plot y en el volcano plot?
> 2. ¿Los genes del heatmap separan las condiciones?
> 3. ¿Los genes canónicos de respuesta a TNF se comportan como esperabas?
> 4. ¿Qué vías biológicas podrían estar afectadas a la vista de los genes más
>    significativos?

---
## Extensión opcional: anotación y enriquecimiento funcional

Esta parte está desactivada por defecto para que el cuaderno principal sea rápido y
reproducible. Si quieres ejecutarla, cambia `EJECUTAR_ENRIQUECIMIENTO = True`.

In [ ]:
# ============================================================================
# CELDA 13 (opcional): Enriquecimiento funcional con Enrichr
# ============================================================================

EJECUTAR_ENRIQUECIMIENTO = False

if EJECUTAR_ENRIQUECIMIENTO:
    import sys
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mygene", "gseapy"], check=True)

    import mygene
    import gseapy as gp

    # Anotar Ensembl IDs → símbolos de gen
    genes_de = resultados_sig.index.tolist()
    mg = mygene.MyGeneInfo()
    annot = mg.querymany(
        genes_de,
        scopes="ensembl.gene",
        fields="symbol",
        species="human",
        as_dataframe=True,
    )
    symbols = annot["symbol"].dropna().astype(str).drop_duplicates().tolist()
    print(f"Genes DE anotados con símbolo: {len(symbols)}")

    enr = gp.enrichr(
        gene_list=symbols,
        gene_sets=["MSigDB_Hallmark_2020", "GO_Biological_Process_2023", "KEGG_2021_Human"],
        organism="human",
        outdir=str(OUT_DIR / "enrichr"),
        cutoff=0.05,
    )
    display(enr.results.sort_values("Adjusted P-value").head(20))
    enr.results.to_csv(OUT_DIR / "enriquecimiento_enrichr.csv", index=False)
else:
    print("Enriquecimiento opcional desactivado. Cambia EJECUTAR_ENRIQUECIMIENTO a True para ejecutarlo.")

---
## Exportación de resultados

In [ ]:
# ============================================================================
# CELDA 14: Comprimir y descargar resultados
# ============================================================================
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/tnf_t3m4_results", "zip", OUT_DIR)
print(f"✓ ZIP creado: {zip_path}")
files.download(zip_path)

---
## Recordatorio de entrega

Cada estudiante debe entregar:

1. **`analisis_RNAseq.py`** — versión ejecutable del análisis. En Colab puedes obtenerla
   con **Archivo → Descargar → Descargar .py** y revisarla en Spyder.
2. **`informe_RNAseq.pdf`** — respuestas a las preguntas, tablas principales, figuras
   con leyenda y conclusiones biológicas fundamentadas.